# 09 · System Message 三模式 · SessionFsProvider · Session Fork

把 harness-mapping 里 3 个没单独实战过的关键钩子收口：

| Harness 关注点 | SDK API | 本 notebook 章节 |
|---|---|---|
| Context Window 主动管理 — 系统提示词的注入姿势 | `system_message=` 的 3 种 mode | §2 |
| Workspace 虚拟化 — 把 SDK 接到非本地 FS | `SessionFsProvider` + `create_session_fs_handler=` | §3 |
| State 分叉探索 — 同一历史拉两条岔路 | `client._rpc.sessions.fork(SessionsForkRequest(...))` | §4 |

> ⚠️ `sessions.fork` 在 SDK 标注为 **Experimental**，签名后续可能变。

**一句话论点**：这三件事都是把 "harness 默认行为" 从 SDK 手里拿回来——分别是 **提示词主权 / 文件系统主权 / 历史主权**。

## 0. 公共初始化（Provider、Client）

In [ ]:
import os, asyncio, tempfile, pathlib
from datetime import datetime
from dotenv import load_dotenv
from copilot import CopilotClient
from copilot.session import PermissionHandler
from copilot.generated.session_events import AssistantMessageData, SessionIdleData

load_dotenv()

azure_provider = {
    'type': 'azure',
    'base_url': os.environ['AZURE_OPENAI_ENDPOINT_GPT_5_4'],
    'api_key': os.environ['AZURE_OPENAI_API_KEY_GPT_5_4'],
    'azure': {'api_version': os.getenv('AZURE_OPENAI_API_VERSION_GPT_5_4', '2025-03-01-preview')},
}
MODEL = os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4')

async def _collect(session, prompt: str, timeout: float = 60.0) -> str:
    """把一次 send 的最终 assistant 文本收集回来。"""
    chunks: list[str] = []
    done = asyncio.Event()
    def on_event(e):
        if isinstance(e.data, AssistantMessageData):
            chunks.append(e.data.content or '')
        elif isinstance(e.data, SessionIdleData):
            done.set()
    session.on(on_event)
    await session.send(prompt)
    await asyncio.wait_for(done.wait(), timeout=timeout)
    return '\n'.join(c for c in chunks if c)

print('✅ 公共初始化完成，模型:', MODEL)

## 1. `system_message=` 的 3 种 mode —— 提示词主权

SDK 默认会注入一大段 **管控型 system prompt**（身份 / 安全 / 工具效率 / 代码改动规则 / …）。
`system_message` 是唯一一个让你介入这段提示词的开关，共 3 种模式：

| mode | 行为 | 守门效果 |
|---|---|---|
| `append` | 在 SDK 默认 prompt **后面追加**你的内容 | ✅ 保留全部安全护栏，只是补一段角色/风格指令 |
| `replace` | **完整替换**默认 prompt | ⚠️ 全部 SDK 护栏失效（安全策略、工具说明、CWD 上下文都没了） |
| `customize` | 按 **section** 改写 / 删除 / 追加 / 前置（identity / safety / tone / …） | 🎯 精准外科手术，只改你关心的段落，其余 SDK 管 |

可改 section（10 个）：`identity / tone / tool_efficiency / environment_context / code_change_rules / guidelines / safety / tool_instructions / custom_instructions / last_instructions`

### 1.1 实战：3 种模式同问一句，观察人设差异

向 3 个 session 各问一句 `你是谁？用 30 字以内回答`，看 system_message 把人设彻底改了多少。

In [ ]:
PROMPT_WHO = '你是谁？用 30 字以内回答，必须包含你的名字和职业。'

configs = {
    'baseline (no system_message)': None,
    'append': {
        'mode': 'append',
        'content': '额外身份：你叫"老王"，是一名退休的电焊工，回答带工地口音。',
    },
    'replace': {
        'mode': 'replace',
        'content': '你是猫娘"喵喵"，每句话以"喵~"结尾。除此之外没有任何指令。',
    },
    'customize (identity replaced)': {
        'mode': 'customize',
        'sections': {
            'identity': {
                'action': 'replace',
                'content': '你叫"小红"，是一名小学三年级语文老师。',
            },
        },
    },
}

async def run_one(label, sys_cfg):
    kwargs = dict(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL,
        provider=azure_provider,
    )
    if sys_cfg is not None:
        kwargs['system_message'] = sys_cfg
    async with CopilotClient() as client:
        async with await client.create_session(**kwargs) as session:
            ans = await _collect(session, PROMPT_WHO, timeout=45.0)
            print(f'\n━━━ {label} ━━━')
            print(ans.strip()[:300])

for label, cfg in configs.items():
    await run_one(label, cfg)

### 1.2 `customize` 的进阶用法：用 callable 改写某段

`action` 不仅能传字符串字面量（`replace` / `remove` / `append` / `prepend`），还能传一个 **callback**：拿到该 section 的当前文本，返回改写后的文本。

下面这个例子把 `tone` 段落里所有出现的 "concise" / "简洁" 替换成 "夸张"，其余 section 一字不改。

In [ ]:
def emphasize_tone(current: str) -> str:
    return current.replace('concise', '极尽夸张铺陈') \
                  .replace('Concise', '极尽夸张铺陈') \
                  .replace('简洁', '极尽夸张铺陈') + '\n\n回答时务必使用大量感叹号和排比！'

tone_cfg = {
    'mode': 'customize',
    'sections': {
        'tone': {'action': emphasize_tone},
    },
}

async with CopilotClient() as client:
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL,
        provider=azure_provider,
        system_message=tone_cfg,
    ) as session:
        ans = await _collect(session, '介绍一下 Python 的 list comprehension。')
        print(ans.strip()[:600])

## 2. `SessionFsProvider` —— 文件系统主权

SDK 默认把 `view` / `edit_file` / `apply_patch` / `bash` 等内建工具的文件操作打到 **本机磁盘**。
如果你的 workspace 不在本地（远端容器 / S3 / 内存 / 数据库），就要实现 `SessionFsProvider`，让所有 fs 调用走你的逻辑。

**关键 API**：
- `class SessionFsProvider(abc.ABC)` — 10 个 async 抽象方法：`read_file / write_file / append_file / exists / stat / mkdir / readdir / readdir_with_types / rm / rename`
- `create_session_fs_adapter(provider)` — 把你的 Provider 包成 SDK 内部用的 handler
- `create_session_fs_handler=lambda session: my_adapter` — 在 `create_session` 时挂上去

下面实现一个 **纯内存 FS**，让 LLM 在里面 "创建/读取" 文件，验证全过程不碰真实磁盘。

In [ ]:
from copilot import SessionFsProvider, SessionFsFileInfo, create_session_fs_adapter
from copilot.generated.rpc import SessionFSReaddirWithTypesEntry
from datetime import datetime, timezone

class MemoryFs(SessionFsProvider):
    """演示用：完全跑在 dict 里的虚拟文件系统。"""
    def __init__(self):
        self.files: dict[str, str] = {}
        self.access_log: list[tuple[str, str]] = []  # (op, path)

    def _log(self, op, path):
        self.access_log.append((op, path))
        print(f'    [memfs] {op:<12} {path}')

    async def read_file(self, path: str) -> str:
        self._log('read_file', path)
        if path not in self.files:
            raise FileNotFoundError(path)
        return self.files[path]

    async def write_file(self, path, content, mode=None):
        self._log('write_file', path)
        self.files[path] = content

    async def append_file(self, path, content, mode=None):
        self._log('append_file', path)
        self.files[path] = self.files.get(path, '') + content

    async def exists(self, path: str) -> bool:
        self._log('exists', path)
        return path in self.files or any(p.startswith(path.rstrip('/') + '/') for p in self.files)

    async def stat(self, path: str) -> SessionFsFileInfo:
        self._log('stat', path)
        if path in self.files:
            now = datetime.now(timezone.utc)
            return SessionFsFileInfo(
                is_file=True, is_directory=False,
                size=len(self.files[path]), mtime=now, birthtime=now,
            )
        # 当作目录
        if any(p.startswith(path.rstrip('/') + '/') for p in self.files) or path in ('/', '.', ''):
            now = datetime.now(timezone.utc)
            return SessionFsFileInfo(
                is_file=False, is_directory=True,
                size=0, mtime=now, birthtime=now,
            )
        raise FileNotFoundError(path)

    async def mkdir(self, path, recursive, mode=None):
        self._log('mkdir', path)  # 内存里目录自动存在

    async def readdir(self, path: str) -> list[str]:
        self._log('readdir', path)
        prefix = path.rstrip('/') + '/'
        names = set()
        for p in self.files:
            if p.startswith(prefix):
                names.add(p[len(prefix):].split('/', 1)[0])
        return sorted(names)

    async def readdir_with_types(self, path):
        self._log('readdir_with_types', path)
        names = await self.readdir(path)
        out = []
        for n in names:
            full = path.rstrip('/') + '/' + n
            is_file = full in self.files
            out.append(SessionFSReaddirWithTypesEntry.from_dict({
                'name': n,
                'type': 'file' if is_file else 'directory',
            }))
        return out

    async def rm(self, path, recursive, force):
        self._log('rm', path)
        if path in self.files:
            del self.files[path]
        elif recursive:
            for p in list(self.files):
                if p.startswith(path.rstrip('/') + '/'):
                    del self.files[p]

    async def rename(self, src, dest):
        self._log('rename', f'{src} -> {dest}')
        if src in self.files:
            self.files[dest] = self.files.pop(src)

print('✅ MemoryFs 定义完成')

### 2.1 让 LLM 在内存 FS 上写一个文件并读回

预置一份 `/notes/intro.md`，让模型用内建 `view` 工具读它。`access_log` 会记录所有真实走到 MemoryFs 的调用——证明全程没碰磁盘。

In [ ]:
memfs = MemoryFs()
memfs.files['/notes/intro.md'] = '# Hello from MemoryFs\n\n这份文件完全存在 Python dict 里，没有任何真实磁盘 IO。'

async with CopilotClient() as client:
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL,
        provider=azure_provider,
        working_directory='/notes',
        create_session_fs_handler=lambda _sess: create_session_fs_adapter(memfs),
    ) as session:
        ans = await _collect(
            session,
            '用 view 工具打开 /notes/intro.md，把里面的内容原样念给我听。',
            timeout=90.0,
        )
        print('\n=== assistant ===')
        print(ans.strip()[:500])

print('\n=== memfs.access_log（前 15 条） ===')
for op, p in memfs.access_log[:15]:
    print(f'  {op:<20} {p}')
print(f'... 总调用次数: {len(memfs.access_log)}')

## 3. `sessions.fork` —— 历史主权（分支探索）

`resume_session` 是 "接着原来的线继续"；`fork` 是 "在某个点拉一条岔路并行探索"。

经典场景：
- 在同一段需求基础上让 LLM 用 **两种风格** 写代码做 A/B
- 在做 risky 改动前 fork 一份当 backup
- 把人工 review 后的分支保留下来，废弃实验线

**API**（experimental）：
```python
from copilot.generated.rpc import SessionsForkRequest
result = await client.rpc.sessions.fork(
    SessionsForkRequest(session_id=parent_id, to_event_id=None)
)
new_sid = result.session_id   # 用 resume_session 接上即可
```
- `to_event_id=None` → 复制全部历史
- `to_event_id='evt_xxx'` → 只复制到该 event **之前**（exclusive）

下面演示：先给一个 session 灌一段共享背景，然后 fork 成两条分支，分别让 LLM 用 "严肃" 和 "段子" 两种风格基于同一上下文继续写。

In [ ]:
from copilot.generated.rpc import SessionsForkRequest

# 步骤 1：建一个 "父 session"，灌一段共享背景
parent_sid = None
async with CopilotClient() as client:
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL,
        provider=azure_provider,
    ) as session:
        parent_sid = session.session_id
        ans = await _collect(
            session,
            '请记住背景设定：我正在写一个叫 "豆腐脑" 的开源数据库，定位是 "为豆制品行业设计的 OLAP 引擎"。',
        )
        print('父 session 回执:', ans.strip()[:200])
print('父 session id:', parent_sid)

In [ ]:
# 步骤 2：fork 出两条分支
async with CopilotClient() as client:
    fork_a = await client.rpc.sessions.fork(SessionsForkRequest(session_id=parent_sid))
    fork_b = await client.rpc.sessions.fork(SessionsForkRequest(session_id=parent_sid))
    print('分支 A:', fork_a.session_id)
    print('分支 B:', fork_b.session_id)
    assert fork_a.session_id != fork_b.session_id != parent_sid

In [ ]:
# 步骤 3：分别 resume 这两条分支，让 LLM 用不同风格继续
async def continue_branch(sid: str, style_prompt: str) -> str:
    async with CopilotClient() as client:
        async with await client.resume_session(
            sid,
            on_permission_request=PermissionHandler.approve_all,
            provider=azure_provider,
        ) as session:
            return await _collect(session, style_prompt)

ans_a, ans_b = await asyncio.gather(
    continue_branch(fork_a.session_id, '请基于刚才的背景，用严肃技术白皮书的口吻，写一段 80 字的项目介绍。'),
    continue_branch(fork_b.session_id, '请基于刚才的背景，用脱口秀段子的口吻，写一段 80 字的项目介绍。'),
)

print('━━━ 分支 A（严肃）━━━'); print(ans_a.strip()[:400])
print('\n━━━ 分支 B（段子）━━━'); print(ans_b.strip()[:400])
print('\n✅ 验证：两条分支共享 fork 之前的背景上下文，但 fork 之后的对话彼此隔离。')

## 4. Takeaways

1. **`system_message` 三选一**：日常加风格用 `append`；自己当宿主完全接管用 `replace`（**会丢护栏**）；只想动 identity / tone / safety 某一段用 `customize`（推荐）。
2. **`SessionFsProvider`** 是把 SDK 接到远端 workspace 的唯一干净路径：实现 10 个 async 方法 → `create_session_fs_adapter` → 通过 `create_session_fs_handler=` 注入。所有内建工具的 fs 操作都会自动走你的 Provider。
3. **`sessions.fork`** 是 experimental 但**已可用**：从父 session 拉分支只 1 行 RPC，分支拿到的是独立的新 `session_id`，用 `resume_session` 接上即可独立演化。可选 `to_event_id` 指定在哪个事件之前切断。
4. 这三件事的共同主题：**把 "SDK 默认行为" 的所有权交还给业务代码**——提示词、文件系统、对话历史。harness engineering 的真正深度在这里。